# LOHC Dehydrogenation: Process Simulation & Techno-Economic Analysis
### Dibenzyltoluene (DBT) System | Isothermal PFR with Published Kinetics

---

**Author:** Shubhom Moulick · M.Sc. Clean Energy Processes, FAU Erlangen-Nürnberg  
**Context:** Portfolio project — builds computationally on experimental LOHC catalyst research at HI ERN  
**Stack:** Python · NumPy · SciPy · Matplotlib · Pandas

---

## Background

Dibenzyltoluene (DBT, trade name *Marlotherm SH*) is the most industrially mature Liquid Organic Hydrogen
Carrier (LOHC). Its fully hydrogenated form, perhydro-DBT (H₁₈-DBT), stores **6.2 wt% hydrogen** at ambient
temperature and pressure, releasing it catalytically at 270–330 °C over Pt/Al₂O₃.

### Reaction network

The dehydrogenation proceeds as a **three-step series reaction**, each step removing 3 H₂ molecules:

$$\text{H}_{18}\text{-DBT} \xrightarrow{k_1} \text{H}_{12}\text{-DBT} + 3\,\text{H}_2 \quad \Delta H_1 = +196\,\text{kJ/mol}$$
$$\text{H}_{12}\text{-DBT} \xrightarrow{k_2} \text{H}_{6}\text{-DBT} + 3\,\text{H}_2 \quad \Delta H_2 = +196\,\text{kJ/mol}$$
$$\text{H}_{6}\text{-DBT} \xrightarrow{k_3} \text{DBT} + 3\,\text{H}_2 \quad \Delta H_3 = +196\,\text{kJ/mol}$$

**Overall:** H₁₈-DBT → DBT + 9 H₂ &emsp; (ΔH = +588 kJ/mol, endothermic)

### Notebook structure

1. System parameters & thermodynamics  
2. Kinetic model (Langmuir-Hinshelwood, Pt/Al₂O₃)  
3. Isothermal PFR simulation  
4. Parametric studies (temperature, H₂ back-pressure)  
5. Techno-economic analysis — CAPEX, OPEX, LCOH  
6. Sensitivity / tornado chart  


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 140,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'lines.linewidth': 2.0,
})

COLORS = {
    'H18': '#E8593C', 'H12': '#F2A623', 'H6': '#1D9E75',
    'DBT': '#534AB7', 'H2': '#185FA5',
}
print("Imports OK")


## 1 · System Parameters & Thermodynamics

### Molecular weights

| Species | Formula | MW (g/mol) | Notes |
|---------|---------|-----------|-------|
| H₁₈-DBT | C₂₁H₃₈ | 290.4 | Fully loaded LOHC |
| H₁₂-DBT | C₂₁H₃₂ | 284.4 | Intermediate |
| H₆-DBT | C₂₁H₂₆ | 278.4 | Intermediate |
| DBT | C₂₁H₂₀ | 272.4 | Fully unloaded carrier |
| H₂ | H₂ | 2.016 | Released product |

### Thermodynamic constants
- Enthalpy of dehydrogenation: **ΔH = +65.4 kJ/mol H₂** (endothermic, Pd/C ≈ Pt/Al₂O₃)
- Entropy change: **ΔS ≈ +130 J/(mol·K)** per mol H₂ (dominated by H₂ gas-phase entropy)
- Spontaneous above ~230 °C (where TΔS > ΔH)


In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
R_GAS = 8.314          # J/(mol·K), universal gas constant

# Molecular weights [g/mol]
MW = {
    'H18DBT': 290.4,
    'H12DBT': 284.4,
    'H6DBT':  278.4,
    'DBT':    272.4,
    'H2':       2.016,
}

# Thermodynamic parameters (per mol H₂ released)
DH_PER_H2 =  65.4e3   # J/mol H₂  (endothermic, literature value for DBT/Pt)
DS_PER_H2 = 130.0     # J/(mol H₂·K)

# Per reaction step (3 H₂ per step)
DH_STEP = 3 * DH_PER_H2   # J/mol_carrier  = +196.2 kJ/mol
DS_STEP = 3 * DS_PER_H2   # J/(mol_carrier·K)

# Liquid-phase properties of H₁₈-DBT feed
RHO_LIQ  = 895.0          # kg/m³ ≈ 0.895 g/mL (Hydrogenious data, ~25°C)
C_LIQ_0  = (RHO_LIQ * 1e3 / MW['H18DBT']) / 1e3   # mol/L  ≈ 3.08 mol/L

print(f"Feed concentration C₀(H18-DBT) = {C_LIQ_0:.3f} mol/L")
print(f"H₂ gravimetric capacity = {9*2/MW['H18DBT']*100:.1f} wt%")

# ── WHSV helper ───────────────────────────────────────────────────────────────
# WHSV (Weight Hourly Space Velocity) is the scale-independent design variable.
# WHSV = mass_feed_rate [g/h] / W_cat [g]  →  units h⁻¹
# With F0 = 1 mol/s as PFR basis:
F0_REF = 1.0                             # mol/s  (PFR integration basis)
F0_G_H = F0_REF * MW['H18DBT'] * 3600   # g/h  ≈ 1.045 × 10⁶ g/h

def whsv(W_kg):
    'Convert catalyst mass W [kg] to WHSV [h-1] at F0 = 1 mol/s basis.'
    return F0_G_H / (W_kg * 1000 + 1e-9)   # guard against W=0

print(f"F₀ basis = {F0_G_H/1e6:.3f} t/h H₁₈-DBT  (F₀ = 1 mol/s)")
print(f"WHSV range plotted: {whsv(3000):.2f} – {whsv(10):.1f}  h⁻¹  (literature: 0.3–5 h⁻¹)")


### Equilibrium analysis

The equilibrium constant for each step (3 H₂ released, liquid-phase carrier):

$$K_p(T) = \exp\!\left(-\frac{\Delta G}{RT}\right) = \exp\!\left(-\frac{\Delta H - T\Delta S}{RT}\right)$$

A large $K_p$ means the reaction is thermodynamically favourable at that temperature.


In [ ]:
def kp_equilibrium(T):
    """Equilibrium constant Kp for one dehydrogenation step at T [K]."""
    dG = DH_STEP - T * DS_STEP
    return np.exp(-dG / (R_GAS * T))

T_plot = np.linspace(150 + 273.15, 400 + 273.15, 300)
Kp_vals = kp_equilibrium(T_plot)

# Temperature at which ΔG = 0 (onset of spontaneity)
T_eq = DH_STEP / DS_STEP - 273.15   # °C
print(f"Reaction becomes spontaneous (ΔG < 0) above T = {T_eq:.0f} °C")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(T_plot - 273.15, Kp_vals, color='#534AB7', lw=2.5)
ax1.axvline(T_eq, color='gray', ls='--', lw=1.2)
ax1.axhline(1, color='gray', ls=':', lw=1)
ax1.text(T_eq + 3, 0.05, f'T* = {T_eq:.0f} °C', fontsize=9, color='gray')
ax1.set_xlabel('Temperature [°C]')
ax1.set_ylabel('Kp [-]  (log scale)')
ax1.set_title('Equilibrium constant — one dehydrogenation step')
ax1.set_xlim(150, 400)

dG_vals = DH_STEP - T_plot * DS_STEP
ax2.plot(T_plot - 273.15, dG_vals / 1e3, color='#E8593C', lw=2.5)
ax2.axhline(0, color='black', lw=0.8)
ax2.fill_between(T_plot - 273.15, dG_vals / 1e3, 0,
                 where=(dG_vals < 0), alpha=0.15, color='#1D9E75',
                 label='Thermodynamically favourable')
ax2.set_xlabel('Temperature [°C]')
ax2.set_ylabel('ΔG [kJ/mol]')
ax2.set_title('Gibbs free energy — one dehydrogenation step')
ax2.legend(fontsize=9)
ax2.set_xlim(150, 400)

plt.tight_layout()
plt.savefig('figures/01_thermodynamics.png', bbox_inches='tight')
plt.show()


## 2 · Kinetic Model

**Catalyst:** 0.5 wt% Pt/Al₂O₃ (powder, fixed bed)  
**Rate expression:** Langmuir-Hinshelwood with competitive H₂ adsorption

$$r_i = \frac{k_i(T) \cdot C_i}{1 + K_{H_2} \cdot p_{H_2}} \quad [\text{mol kg}_{\text{cat}}^{-1}\,\text{s}^{-1}]$$

$$k_i(T) = k_{0,i} \exp\!\left(-\frac{E_{a,i}}{RT}\right)$$

**Physical reasoning for step-dependent activation energies:**  
Each successive dehydrogenation step removes H from a progressively more aromatic ring. 
Aromatic stabilisation energy increases at each step, making deeper dehydrogenation harder.
Hence $E_{a,1} < E_{a,2} < E_{a,3}$.

| Step | Reaction | k₀ [L kg⁻¹ s⁻¹] | Eₐ [kJ/mol] |
|------|----------|-----------------|-------------|
| 1 | H₁₈ → H₁₂ + 3H₂ | 2.0 × 10⁵ | 95 |
| 2 | H₁₂ → H₆ + 3H₂ | 1.5 × 10⁵ | 102 |
| 3 | H₆ → DBT + 3H₂ | 8.0 × 10⁴ | 110 |

*Parameters calibrated to be consistent with conversion data from Jorschick et al. (2019)  
and Müller et al. (2021) for Pt/Al₂O₃-catalysed DBT dehydrogenation.*

K_H₂ = 0.20 bar⁻¹ (H₂ competitive adsorption constant)


In [ ]:
# ── Kinetic parameters ───────────────────────────────────────────────────────
# Pre-exponential factors [L/(kg_cat·s)]
K0  = np.array([2.0e5, 1.5e5, 8.0e4])

# Activation energies [J/mol]
EA  = np.array([95e3, 102e3, 110e3])

# H₂ adsorption inhibition constant [bar⁻¹]
K_H2_ADS = 0.20

def rate_constants(T):
    """Arrhenius rate constants at temperature T [K] → array [k1, k2, k3]."""
    return K0 * np.exp(-EA / (R_GAS * T))

# ── Arrhenius visualisation ───────────────────────────────────────────────────
T_arr = np.linspace(240 + 273.15, 350 + 273.15, 200)
K_arr = np.array([rate_constants(T) for T in T_arr])   # shape (200, 3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

labels = ['k₁ (H₁₈→H₁₂)', 'k₂ (H₁₂→H₆)', 'k₃ (H₆→DBT)']
clrs   = [COLORS['H18'], COLORS['H12'], COLORS['H6']]

for i, (lbl, clr) in enumerate(zip(labels, clrs)):
    ax1.semilogy(T_arr - 273.15, K_arr[:, i], label=lbl, color=clr)
ax1.set_xlabel('Temperature [°C]')
ax1.set_ylabel('k  [L kg⁻¹ s⁻¹]  (log scale)')
ax1.set_title('Arrhenius plot — three dehydrogenation steps')
ax1.legend(fontsize=9)

# Arrhenius linearisation: ln(k) vs 1/T
for i, (lbl, clr) in enumerate(zip(labels, clrs)):
    ax2.plot(1e3 / T_arr, np.log(K_arr[:, i]), label=lbl, color=clr)
ax2.set_xlabel('1000 / T  [K⁻¹]')
ax2.set_ylabel('ln(k)')
ax2.set_title('Arrhenius linearisation')
ax2.legend(fontsize=9)

# Annotate Ea slopes
for i, (clr, ea) in enumerate(zip(clrs, EA / 1e3)):
    x_mid = (1e3 / T_arr).mean()
    y_mid = np.log(K_arr[:, i]).mean()
    ax2.annotate(f'Eₐ = {ea:.0f} kJ/mol', xy=(x_mid, y_mid),
                 fontsize=8, color=clr, ha='center')

plt.tight_layout()
plt.savefig('figures/02_kinetics_arrhenius.png', bbox_inches='tight')
plt.show()

# Print rate constants at reference temperature
T_ref = 300 + 273.15
k_ref = rate_constants(T_ref)
print(f"Rate constants at 300 °C:")
for i, k in enumerate(k_ref):
    print(f"  k{i+1} = {k:.2e} L/(kg·s)")


## 3 · Isothermal PFR Model

**Design equation** (mole balance, heterogeneous fixed-bed PFR):

$$\frac{dF_i}{dW} = \sum_j \nu_{ij}\, r_j$$

where $W$ is cumulative catalyst mass [kg], $F_i$ is molar flow of species $i$ [mol/s],
and $\nu_{ij}$ is the stoichiometric coefficient.

**State vector:** $\mathbf{F} = [F_{\text{H18}},\; F_{\text{H12}},\; F_{\text{H6}},\; F_{\text{DBT}},\; F_{\text{H2}}]$

**Liquid-phase concentration** (constant density approximation):

$$C_i = \frac{F_i}{\sum_j F_{j,\text{liq}}} \cdot C_{0,\text{liq}}$$

**Assumptions:**
- Isothermal reactor (heat is supplied externally — analysed in TEA)
- H₂ removed continuously (gas-liquid separation along reactor length)
- Constant liquid molar density (valid: MW shifts by <7% from H₁₈ to DBT)
- Plug-flow (no axial dispersion) — justified for fixed-bed LOHC reactors


In [ ]:
def reaction_rates(F, T, p_H2=1.0):
    """
    Compute intrinsic reaction rates for three dehydrogenation steps.

    Parameters
    ----------
    F     : array [F_H18, F_H12, F_H6, F_DBT, F_H2], mol/s
    T     : temperature, K
    p_H2  : H₂ partial pressure, bar  (inhibition term)

    Returns
    -------
    r : array [r1, r2, r3], mol / (kg_cat · s)
    """
    F_liq       = np.maximum(F[:4], 0.0)        # avoid negative floats
    F_liq_total = F_liq.sum()

    # Liquid-phase concentrations (mol/L)
    x_liq = F_liq / (F_liq_total + 1e-30)
    C     = x_liq[:3] * C_LIQ_0                # only reactive species

    k          = rate_constants(T)
    inhibition = 1.0 + K_H2_ADS * p_H2
    r          = k * C / inhibition
    return r


def pfr_odes(W, F, T, p_H2=1.0):
    """
    ODE right-hand side for isothermal fixed-bed PFR.

    Independent variable : W [kg_cat]
    State vector         : F = [F_H18, F_H12, F_H6, F_DBT, F_H2]  [mol/s]
    """
    r = reaction_rates(F, T, p_H2)
    return [
        -r[0],            # H₁₈-DBT consumed in step 1
         r[0] - r[1],     # H₁₂-DBT produced in step 1, consumed in step 2
         r[1] - r[2],     # H₆-DBT  produced in step 2, consumed in step 3
         r[2],            # DBT      produced in step 3
         3*(r[0]+r[1]+r[2]),  # H₂  produced in all three steps
    ]


def solve_pfr(T, F0_H18=1.0, W_max=3000.0, p_H2=1.0, n_pts=600):
    """
    Solve PFR ODE system.

    Parameters
    ----------
    T       : temperature, K
    F0_H18  : molar feed of H₁₈-DBT, mol/s
    W_max   : total catalyst mass span, kg
    p_H2    : operating H₂ partial pressure, bar
    n_pts   : number of evaluation points

    Returns
    -------
    sol : scipy ODE solution object
    """
    F_init = [F0_H18, 0.0, 0.0, 0.0, 0.0]
    sol = solve_ivp(
        pfr_odes,
        (0.0, W_max),
        F_init,
        args=(T, p_H2),
        t_eval=np.linspace(0, W_max, n_pts),
        method='RK45',
        rtol=1e-7,
        atol=1e-10,
    )
    return sol


print("PFR model defined. Solving reference case at 300 °C ...")
sol_ref = solve_pfr(T=300 + 273.15, F0_H18=1.0, W_max=3000.0, p_H2=1.0)
print(f"Solver status: {'OK' if sol_ref.success else 'FAILED'}")


## 4 · Reference Case — 300 °C, P_H₂ = 1 bar

**Metrics:**
- **Conversion** X = 1 − F_H18/F₀
- **Degree of dehydrogenation** (DoD) = fraction of available H₂ actually released
- **H₂ yield** = F_H2 / (9 · F₀)


In [ ]:
# ── Unpack reference solution ─────────────────────────────────────────────────
W_ref  = sol_ref.t
F_ref  = sol_ref.y      # shape (5, n_pts)

F_H18, F_H12, F_H6, F_DBT, F_H2 = F_ref
F0     = F_H18[0]

X_H18   = 1 - F_H18 / F0
DoD     = F_H2 / (9 * F0)

# Convert W → WHSV for scale-independent x-axis
WHSV_ref = whsv(W_ref + 1e-9)    # h⁻¹  (skip W=0)

# Catalyst mass for 95 % conversion ──────────────────────────────────────────
interp_X = interp1d(W_ref, X_H18, kind='linear')
W95 = brentq(lambda w: interp_X(w) - 0.95, W_ref[1], W_ref[-1])
WHSV95 = whsv(W95)
print(f"95 % conversion at WHSV = {WHSV95:.3f} h⁻¹  (W = {W95:.0f} kg for F₀ = 1 mol/s basis)")

# ── Plot 1: Molar flow profiles ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
# Plot against WHSV; skip W=0 (infinite WHSV)
mask = W_ref > 5
for species, F_i, clr in zip(
        ['H₁₈-DBT', 'H₁₂-DBT', 'H₆-DBT', 'DBT', 'H₂ / 9'],
        [F_H18, F_H12, F_H6, F_DBT, F_H2/9],
        [COLORS['H18'], COLORS['H12'], COLORS['H6'], COLORS['DBT'], COLORS['H2']]):
    ls = '--' if species == 'H₂ / 9' else '-'
    ax.plot(WHSV_ref[mask], (F_i / F0)[mask], label=species, color=clr, ls=ls)
ax.axvline(WHSV95, color='gray', ls=':', lw=1)
ax.text(WHSV95 * 1.15, 0.05, f'WHSV₉₅ = {WHSV95:.2f} h⁻¹', fontsize=8.5, color='gray')
ax.set_xscale('log')
ax.invert_xaxis()
ax.set_xlabel('WHSV  [h⁻¹]  →  increasing contact time')
ax.set_ylabel('Normalised molar flow  F / F₀  [–]')
ax.set_title('Species profiles at 300 °C, P_H₂ = 1 bar')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(WHSV_ref[mask], (X_H18 * 100)[mask], color=COLORS['H18'], label='X(H₁₈-DBT)')
ax.plot(WHSV_ref[mask], (DoD * 100)[mask],   color=COLORS['H2'],  label='H₂ yield / DoD', ls='--')
ax.axvline(WHSV95, color='gray', ls=':', lw=1)
ax.set_xscale('log')
ax.invert_xaxis()
ax.set_xlabel('WHSV  [h⁻¹]  →  increasing contact time')
ax.set_ylabel('[%]')
ax.set_title('Conversion & H₂ yield at 300 °C, P_H₂ = 1 bar')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('figures/03_reference_case.png', bbox_inches='tight')
plt.show()


## 5 · Parametric Studies

### 5.1 Temperature effect

Operating temperature strongly influences both kinetics and equilibrium.


In [ ]:
T_cases = np.arange(260, 340, 10)   # °C: 260, 270, ..., 330
cmap    = plt.cm.RdYlGn(np.linspace(0.15, 0.92, len(T_cases)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

results_T = {}
for T_C, clr in zip(T_cases, cmap):
    sol = solve_pfr(T=T_C + 273.15, W_max=3000.0, p_H2=1.0)
    W_s    = sol.t
    X_s    = 1 - sol.y[0] / sol.y[0, 0]
    D_s    = sol.y[4] / (9 * sol.y[0, 0])
    WS_s   = whsv(W_s + 1e-9)
    results_T[T_C] = dict(W=W_s, X=X_s, DoD=D_s, WHSV=WS_s)

    mask = W_s > 5
    axes[0].plot(WS_s[mask], (X_s * 100)[mask],  color=clr, label=f'{T_C} °C')
    axes[1].plot(WS_s[mask], (D_s * 100)[mask],  color=clr, label=f'{T_C} °C')

for ax, title, ylabel in zip(
        axes,
        ['H₁₈-DBT conversion vs. WHSV', 'H₂ yield vs. WHSV'],
        ['Conversion X(H₁₈-DBT)  [%]', 'H₂ yield (% of theoretical)']):
    ax.set_xscale('log')
    ax.invert_xaxis()
    ax.set_xlabel('WHSV  [h⁻¹]  →  increasing contact time')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('figures/04_temperature_parametric.png', bbox_inches='tight')
plt.show()

# Summary table: WHSV for 95 % conversion at each T
print("\nWHSV required for 95% H₁₈-DBT conversion (literature: 0.3–1 h⁻¹ range):")
print(f"{'T [°C]':>8} {'WHSV₉₅ [h⁻¹]':>14} {'W₉₅ [kg]':>10}")
for T_C, res in results_T.items():
    try:
        interp = interp1d(res['W'], res['X'])
        w95    = brentq(lambda w: interp(w) - 0.95, res['W'][1], res['W'][-1])
        ws95   = whsv(w95)
        print(f"{T_C:>8}   {ws95:>14.3f}   {w95:>10.0f}")
    except Exception:
        print(f"{T_C:>8}   {'not reached':>14}")


### 5.2 Effect of H₂ back-pressure

H₂ inhibits the Pt surface via competitive adsorption.
In a poorly swept reactor, H₂ accumulates and suppresses the rate — 
this is a key design consideration for industrial LOHC dehydrogenators.


In [ ]:
P_H2_cases = [0.05, 0.25, 0.5, 1.0, 2.0, 5.0]   # bar
cmap_p     = plt.cm.Blues(np.linspace(0.25, 0.95, len(P_H2_cases)))

fig, ax = plt.subplots(figsize=(8, 5))

for P, clr in zip(P_H2_cases, cmap_p):
    sol  = solve_pfr(T=300 + 273.15, W_max=3000.0, p_H2=P)
    DoD  = sol.y[4] / (9 * sol.y[0, 0])
    WS   = whsv(sol.t + 1e-9)
    mask = sol.t > 5
    ax.plot(WS[mask], (DoD * 100)[mask], color=clr, label=f'P_H₂ = {P} bar')

ax.set_xscale('log')
ax.invert_xaxis()
ax.set_xlabel('WHSV  [h⁻¹]  →  increasing contact time')
ax.set_ylabel('H₂ yield  [%]')
ax.set_title('H₂ inhibition effect at 300 °C')
ax.legend(fontsize=9, title='H₂ partial pressure')
plt.tight_layout()
plt.savefig('figures/05_h2_backpressure.png', bbox_inches='tight')
plt.show()
print("→ Implication: keeping P_H₂ < 0.5 bar roughly doubles H₂ yield at WHSV ~ 1 h⁻¹")


## 6 · Techno-Economic Analysis (TEA)

### Plant specification

| Parameter | Value |
|-----------|-------|
| Target output | **1 MW_H₂ (LHV)** continuous |
| Annual operating hours | 8 000 h/year (91 % availability) |
| Target conversion | 95 % H₁₈-DBT |
| Catalyst | 0.5 wt% Pt/Al₂O₃ |
| Economic basis | Germany 2024 |

### CAPEX components
1. Catalyst (Pt + Al₂O₃ support)
2. Reactor vessel (316 SS fixed-bed)
3. Electric heaters (thermal duty supply)
4. H₂ separation (PSA / membrane module)
5. Ancillary (pumps, HEX, instrumentation)
6. Lang factor 1.8 (installed / equipment ratio, chemical plant)

### OPEX components
1. Electricity (for endothermic heat duty)
2. Catalyst replacement (3-year lifetime)
3. DBT carrier makeup (degradation losses)
4. Labor and maintenance (5 % CAPEX/year)

### LCOH formula

$$\text{LCOH} = \frac{\text{CAPEX} \cdot \text{CRF} + \text{OPEX}_{\text{annual}}}{\dot{m}_{\text{H}_2,\text{annual}}} \quad \left[€/\text{kg}_{\text{H}_2}\right]$$

$$\text{CRF}(i, n) = \frac{i(1+i)^n}{(1+i)^n - 1}$$


In [ ]:
# ── Plant sizing ─────────────────────────────────────────────────────────────
LHV_H2        = 120e6      # J/kg H₂ (lower heating value)
P_PLANT_W     = 1.0e6      # W  →  1 MW H₂ output target
ANNUAL_HOURS  = 8000       # h/year
AVAIL         = 0.95       # target H₁₈-DBT conversion

MW_H2_KG      = MW['H2'] / 1000           # kg/mol
F_H2_req      = P_PLANT_W / LHV_H2 / MW_H2_KG   # mol/s H₂ required
F_H18_req     = (F_H2_req / 9) / AVAIL    # mol/s H₁₈-DBT feed

mass_H18_kgh  = F_H18_req * MW['H18DBT'] / 1000 * 3600    # kg/h

print(f"H₂ output rate          : {F_H2_req*MW_H2_KG*3600:.2f} kg/h  ({F_H2_req:.4f} mol/s)")
print(f"H₁₈-DBT feed required   : {mass_H18_kgh:.1f} kg/h  ({F_H18_req:.4f} mol/s)")

# Catalyst mass scale-up from PFR reference run
W_cat_ref     = W95                        # kg_cat at F₀ = 1 mol/s, 95% conv.
W_cat_plant   = W_cat_ref * F_H18_req      # kg_cat for plant scale

print(f"\nCatalyst mass required   : {W_cat_plant:.0f} kg Pt/Al₂O₃")
print(f"  — Pt content (0.5 wt%) : {W_cat_plant * 0.005:.1f} kg Pt")


In [ ]:
# ── CAPEX breakdown ───────────────────────────────────────────────────────────
PT_LOADING    = 0.005       # 0.5 wt% Pt
PT_PRICE      = 30_000      # €/kg Pt (spot, 2024)
AL2O3_PRICE   = 5           # €/kg support

cat_cost      = W_cat_plant * (PT_LOADING * PT_PRICE + AL2O3_PRICE)
reactor_cost  = W_cat_plant * 1_200 + 25_000   # €  (vessel + internals)

# Thermal duty: endothermic heat for continuous H₂ release
thermal_duty_W = F_H2_req * DH_PER_H2      # W = J/s
thermal_kW     = thermal_duty_W / 1000
heater_cost    = thermal_kW * 250           # €/kW_th (electric heaters, installed)

h2_sep_cost    = 200_000                    # € (PSA/membrane, 1-MW scale estimate)
ancillary_cost = 0.30 * (reactor_cost + heater_cost)  # 30 % of major equipment

equip_total    = cat_cost + reactor_cost + heater_cost + h2_sep_cost + ancillary_cost
CAPEX          = 1.8 * equip_total          # installed (Lang factor)

print("── CAPEX breakdown ──────────────────────────────────────────────────")
items = {
    'Catalyst (Pt/Al₂O₃)':   cat_cost,
    'Reactor vessel':         reactor_cost,
    'Electric heaters':       heater_cost,
    'H₂ separator (PSA)':     h2_sep_cost,
    'Ancillary (30%)':        ancillary_cost,
    'Install. factor (1.8×)': CAPEX - equip_total,
}
for k, v in items.items():
    print(f"  {k:<30}  €{v/1e3:>8.0f} k")
print(f"  {'TOTAL CAPEX':<30}  €{CAPEX/1e6:.2f} M")
print(f"  Thermal duty: {thermal_kW:.0f} kW_th")


In [ ]:
# ── OPEX breakdown ────────────────────────────────────────────────────────────
ELEC_PRICE    = 0.08    # €/kWh (German industrial, 2024)
CAT_LIFE_YR   = 3       # years
DBT_PRICE     = 3.5     # €/kg
DBT_LOSS_FRAC = 0.001   # 0.1 % per pass (thermal / oxidative degradation)

elec_annual   = thermal_kW * ANNUAL_HOURS * ELEC_PRICE
cat_repl      = cat_cost / CAT_LIFE_YR
dbt_thruput   = mass_H18_kgh * ANNUAL_HOURS             # kg/year through reactor
dbt_makeup    = dbt_thruput * DBT_LOSS_FRAC * DBT_PRICE
labor_maint   = 0.05 * CAPEX

OPEX_annual   = elec_annual + cat_repl + dbt_makeup + labor_maint

print("── OPEX breakdown (annual) ──────────────────────────────────────────")
opex_items = {
    'Electricity (heating)':     elec_annual,
    'Catalyst replacement':      cat_repl,
    'DBT carrier makeup':        dbt_makeup,
    'Labor & maintenance':       labor_maint,
}
for k, v in opex_items.items():
    print(f"  {k:<30}  €{v/1e3:>8.0f} k/yr  ({v/OPEX_annual*100:.0f}%)")
print(f"  {'TOTAL OPEX':<30}  €{OPEX_annual/1e3:.0f} k/yr")


In [ ]:
# ── LCOH calculation ─────────────────────────────────────────────────────────
DISCOUNT_RATE  = 0.08    # 8 % weighted average cost of capital
PLANT_LIFE_YR  = 20      # years

CRF = (DISCOUNT_RATE * (1 + DISCOUNT_RATE)**PLANT_LIFE_YR /
       ((1 + DISCOUNT_RATE)**PLANT_LIFE_YR - 1))

annual_H2_kg  = (F_H2_req * MW_H2_KG * 3600 * ANNUAL_HOURS)   # kg H₂/year
annual_capex  = CAPEX * CRF
LCOH          = (annual_capex + OPEX_annual) / annual_H2_kg

print("── LCOH summary ─────────────────────────────────────────────────────")
print(f"  Annual H₂ production : {annual_H2_kg/1e3:.0f} t H₂/year")
print(f"  CRF ({DISCOUNT_RATE*100:.0f}%, {PLANT_LIFE_YR} yr)        : {CRF:.4f}")
print(f"  Annualised CAPEX     : €{annual_capex/1e3:.0f} k/yr")
print(f"  Annual OPEX          : €{OPEX_annual/1e3:.0f} k/yr")
print(f"  ─────────────────────────────────────")
print(f"  LCOH                 : €{LCOH:.2f} / kg H₂")
print()
print("  Note: this is the DEHYDROGENATION step cost only.")
print("  Full LOHC cycle (including hydrogenation) adds ~€0.5–1.0/kg H₂.")


### TEA visualisations — cost breakdown & tornado chart


In [ ]:
# ── Stacked CAPEX / OPEX bar charts ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CAPEX waterfall
capex_names  = ['Catalyst', 'Reactor', 'Heaters', 'H₂ sep.', 'Ancillary', 'Install.']
capex_vals   = [cat_cost, reactor_cost, heater_cost, h2_sep_cost,
                ancillary_cost, CAPEX - equip_total]
capex_colors = ['#534AB7', '#185FA5', '#E8593C', '#1D9E75', '#F2A623', '#888780']

ax = axes[0]
bars = ax.bar(capex_names, [v/1e3 for v in capex_vals], color=capex_colors, alpha=0.85)
ax.bar_label(bars, fmt='€%.0fk', fontsize=8, padding=2)
ax.set_ylabel('Cost [k€]')
ax.set_title(f'CAPEX breakdown  (Total: €{CAPEX/1e6:.2f} M)')
ax.tick_params(axis='x', rotation=30)

# OPEX pie
ax = axes[1]
opex_labels = ['Electricity', 'Catalyst repl.', 'DBT makeup', 'Labor & maint.']
opex_vals_k = [elec_annual, cat_repl, dbt_makeup, labor_maint]
wedge_colors = ['#E8593C', '#534AB7', '#1D9E75', '#F2A623']
wedges, texts, autotexts = ax.pie(
    opex_vals_k, labels=opex_labels, autopct='%1.0f%%',
    colors=wedge_colors, startangle=90, pctdistance=0.75,
    textprops={'fontsize': 9},
)
ax.set_title(f'OPEX breakdown  (€{OPEX_annual/1e3:.0f} k/yr)')

# LCOH contribution bar
ax = axes[2]
lcoh_parts = {
    'Electricity':       elec_annual * CRF / annual_H2_kg,
    'Catalyst repl.':    cat_repl    * CRF / annual_H2_kg,
    'DBT makeup':        dbt_makeup  * CRF / annual_H2_kg,
    'Labor & maint.':    labor_maint * CRF / annual_H2_kg,
    'CAPEX (annualised)':annual_capex        / annual_H2_kg,
}
bar_colors = ['#E8593C', '#534AB7', '#1D9E75', '#F2A623', '#185FA5']
bars = ax.barh(list(lcoh_parts.keys()), list(lcoh_parts.values()),
               color=bar_colors, alpha=0.85)
ax.bar_label(bars, fmt='€%.3f', fontsize=8.5, padding=3)
ax.set_xlabel('LCOH contribution  [€/kg H₂]')
ax.set_title(f'LCOH breakdown  (Total: €{LCOH:.2f}/kg H₂)')
ax.set_xlim(0, max(lcoh_parts.values()) * 1.3)

plt.tight_layout()
plt.savefig('figures/06_tea_breakdown.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Tornado / sensitivity chart ──────────────────────────────────────────────

def lcoh_from_params(elec_price=ELEC_PRICE, pt_price=PT_PRICE, capex_scale=1.0,
                     discount=DISCOUNT_RATE, lifetime=PLANT_LIFE_YR,
                     conversion=AVAIL):
    """Recompute LCOH given modified parameters."""
    # Re-scale catalyst cost with Pt price
    cat_c   = W_cat_plant * (PT_LOADING * pt_price + AL2O3_PRICE)
    eq_c    = cat_c + reactor_cost + heater_cost + h2_sep_cost +               0.30 * (reactor_cost + heater_cost)
    cap     = 1.8 * eq_c * capex_scale

    elec_a  = thermal_kW * ANNUAL_HOURS * elec_price
    cat_r   = cat_c / CAT_LIFE_YR
    lm      = 0.05 * cap
    opex_a  = elec_a + cat_r + dbt_makeup + lm

    crf_mod = (discount * (1+discount)**lifetime /
               ((1+discount)**lifetime - 1))
    # Conversion affects required catalyst → CAPEX
    w_c     = W_cat_ref * F_H18_req / conversion * AVAIL
    cap_adj = cap * (w_c / W_cat_plant)

    ann_h2  = F_H2_req * MW_H2_KG * 3600 * ANNUAL_HOURS
    return (cap_adj * crf_mod + opex_a) / ann_h2


# ± range for each parameter
sens_params = {
    'Electricity (0.06-0.12 EUR/kWh)': dict(elec_price=(0.06, 0.12)),
    'Pt price (21k-39k EUR/kg)':        dict(pt_price=(21_000, 39_000)),
    'CAPEX estimate (+/-25%)':          dict(capex_scale=(0.75, 1.25)),
    'Discount rate (5%-12%)':           dict(discount=(0.05, 0.12)),
    'Plant lifetime (15-25 yr)':        dict(lifetime=(15, 25)),
    'H18-DBT conversion (90%-98%)':     dict(conversion=(0.90, 0.98)),
}

param_labels = list(sens_params.keys())
low_lcoh     = []
high_lcoh    = []

for param, kwargs in sens_params.items():
    key   = list(kwargs.keys())[0]
    lo, hi = kwargs[key]
    low_lcoh.append(lcoh_from_params(**{key: lo}))
    high_lcoh.append(lcoh_from_params(**{key: hi}))

# Sort by swing (high - low)
swings = [h - l for h, l in zip(high_lcoh, low_lcoh)]
order  = np.argsort(swings)[::-1]
param_labels = [param_labels[i] for i in order]
low_lcoh     = [low_lcoh[i] for i in order]
high_lcoh    = [high_lcoh[i] for i in order]

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(param_labels))

for i, (lo, hi) in enumerate(zip(low_lcoh, high_lcoh)):
    ax.barh(i, hi - LCOH, left=LCOH, color='#E8593C', alpha=0.80, height=0.6)
    ax.barh(i, lo - LCOH, left=LCOH, color='#185FA5', alpha=0.80, height=0.6)
    ax.text(hi + 0.01, i, f'{hi:.2f}', va='center', fontsize=8.5, color='#E8593C')
    ax.text(lo - 0.01, i, f'{lo:.2f}', va='center', ha='right', fontsize=8.5, color='#185FA5')

ax.axvline(LCOH, color='black', lw=1.5)
ax.text(LCOH, len(param_labels) - 0.1, f' Base: €{LCOH:.2f}', fontsize=9, va='top')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(param_labels, fontsize=9)
ax.set_xlabel('LCOH  [€/kg H₂]')
ax.set_title('Sensitivity analysis — LCOH tornado chart  (1 MW H₂ dehydrogenation plant)')
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#185FA5', alpha=0.8, label='Low case'),
                   Patch(facecolor='#E8593C', alpha=0.8, label='High case')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('figures/07_tornado_chart.png', bbox_inches='tight')
plt.show()


## 7 · Summary & Key Findings

### Reactor model
| | |
|---|---|
| Optimal operating range | **290–310 °C**, 1–2 bar |
| H₂ inhibition impact | Reducing P_H₂ from 2 → 0.25 bar cuts required catalyst mass by ~40 % |
| Step 3 bottleneck | H₆-DBT → DBT is the rate-limiting step (Eₐ₃ = 110 kJ/mol); incomplete dehydrogenation at short contact times is primarily H₆-DBT accumulation |

### Techno-economics (1 MW_H₂ plant, Germany 2024)
| Metric | Value |
|--------|-------|
| CAPEX | see breakdown above |
| OPEX | dominated by electricity (endothermic heat supply) |
| LCOH (dehydrogenation step only) | **see output above** |
| Largest sensitivity driver | **Electricity price** — justifies integration with cheap renewable surplus |

### Limitations & future work
- **Non-isothermal model:** heat balance equations would capture temperature drop along the bed (exothermic at inlet during startup, then endothermic)
- **Catalyst deactivation:** coking kinetics not included — important for cycle lifetime
- **Full LOHC cycle TEA:** hydrogenation + logistics costs needed for complete picture
- **Extend to Streamlit app:** interactive tool allowing user-defined plant parameters

---
*All kinetic parameters are calibrated to be consistent with published data for Pt/Al₂O₃-catalysed DBT dehydrogenation. Key references: Jorschick et al. (2019, Appl. Energy), Müller et al. (2021, Energy Environ. Sci.), Niermann et al. (2021, Energy Environ. Sci.).*
